# RFP Proposal Generator v2
### One-Shot Learning → Section Generation → IDML Template Injection → PDF Export

**Workflow (run each cell top-to-bottom):**
1. **Setup** — Install packages, configure paths, load firm info
2. **PDF Extraction** — Pull text from the example proposal and the target RFQ
3. **Claude Client** — Initialize with system prompt and cached message builder
4. **Section Generation** — One cell per proposal section; each streams from Claude and returns structured JSON
5. **IDML Injection** — XML helper utilities + story builders map generated JSON → template stories
6. **Build & Verify** — Write the modified IDML, spot-check stories
7. **Export PDF** — AppleScript triggers InDesign to export `example_rfq_to_rfp_proposal.pdf`

> **To use for a new RFQ:** update `RFQ_PDF` in Cell 2 and re-run from top.


In [1]:
%pip install -q anthropic pdfplumber



[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import os, json, re, html, zipfile, subprocess
from pathlib import Path
from datetime import date
import anthropic
import pdfplumber
from IPython.display import display, Markdown

# ── API Key ────────────────────────────────────────────────────────────────────
ANTHROPIC_API_KEY = os.environ.get("ANTHROPIC_API_KEY", "")

# ── Path resolver (handles iCloud Drive / macOS special paths) ─────────────────
def _find(filename):
    """Locate a file by name under the home directory via find."""
    r = subprocess.run(
        ['find', str(Path.home()), '-name', filename,
         '-not', '-path', '*/.Trash/*', '-not', '-path', '*/.git/*'],
        capture_output=True, text=True
    )
    hits = [l for l in r.stdout.strip().split('\n') if l.strip()]
    return Path(hits[0]) if hits else None

# ── File Paths — update RFQ_PDF for each new bid ──────────────────────────────
EXAMPLE_PDF   = _find("RFP 2025-22_Collab Architecture_civic_example_1.pdf")
RFQ_PDF       = _find("RFQ - AE Services for Cascade Campus Renovation Bid 2025-075-Final.pdf")
IDML_TEMPLATE = _find("2026 Master Template_Facing Pages.idml")

_HERE        = Path.cwd()
IDML_OUTPUT  = _HERE / "rfp_filled_template_v2.idml"
PDF_OUTPUT   = _HERE / "example_rfq_to_rfp_proposal.pdf"

# ── Status ─────────────────────────────────────────────────────────────────────
for label, val in [("Example PDF", EXAMPLE_PDF), ("RFQ PDF", RFQ_PDF), ("IDML template", IDML_TEMPLATE)]:
    ok = val and val.exists()
    print(f"{'✓' if ok else '✗'}  {label:15s}: {val}")

print(f"   Output PDF   : {PDF_OUTPUT}")
print(f"   API key      : {'✓ set' if ANTHROPIC_API_KEY else '✗ NOT SET — export ANTHROPIC_API_KEY=sk-ant-...'}")


✓  Example PDF    : /Users/brianpak/Desktop/Desktop - Brian’s MacBook Pro/Projects/Collab Architecture/RFP_Project_2/RFP 2025-22_Collab Architecture_civic_example_1.pdf
✓  RFQ PDF        : /Users/brianpak/Desktop/Desktop - Brian’s MacBook Pro/Projects/Collab Architecture/RFP_Project_2/RFQ - AE Services for Cascade Campus Renovation Bid 2025-075-Final.pdf
✓  IDML template  : /Users/brianpak/Desktop/Desktop - Brian’s MacBook Pro/Projects/Collab Architecture/Templates/InDesign Template_All Files/2026 Master Template_Facing Pages.idml
   Output PDF   : /Users/brianpak/Desktop/Desktop - Brian’s MacBook Pro/Projects/Collab Architecture/RFP_Project_2/RFP_Project_2/example_rfq_to_rfp_proposal.pdf
   API key      : ✓ set


In [3]:
# ── Firm & Team Information ────────────────────────────────────────────────────
# Update this block when submitting for a different firm.

FIRM_INFO = """
FIRM NAME:          Collab Architecture
ADDRESS:            9217 Eastman Park Drive, Unit 3, Windsor, CO 80550
PHONE:              970-292-7078
EMAIL:              jordan@collabarchitects.com
WEBSITE:            www.collabarchitects.com
YEAR FOUNDED:       2020
DISCIPLINES:        Architecture, Interior Design
RECOGNITION:        Named one of the fastest-growing private companies in Northern Colorado
                    and the Front Range (2023 & 2024)

PRIMARY CONTACT:
  Jordan W. Lockner, AIA, NCARB | jordan@collabarchitects.com | 970.215.9907

KEY PERSONNEL:
  - Jordan W. Lockner, AIA, NCARB
      Role:          Principal Architect / Primary Contact
      Education:     University of Colorado, B.ENVD (Architecture focus)
      Registrations: Licensed Architect, NCARB
      Awards:        2022 UofC ENVD Young Designer Award; 2023 N. Colorado 40 Under 40

  - Kala Bailor, AIA, LEED GA
      Role:          Project Manager / Primary Project Contact
      Education:     University of Colorado - Denver, Master of Architecture
      Registrations: Licensed Architect, LEED Green Associate

  - Michael Aller, AIA, LEED AP  ("Mick")
      Role:          QA/QC Manager
      Education:     University of Michigan, Master of Architecture
      Registrations: Licensed Architect, NCARB, LEED Accredited Professional
      Experience:    40+ years in municipal and higher education facility design
      Awards:        AIA Colorado Citation Award; F.W. Dodge Silver Hard Hat Award

SUB-CONSULTANTS:
  LARSEN STRUCTURAL DESIGN - Structural Engineering
      Blake Larsen, PE, LEED AP (19 yrs N. Colorado) | Fort Collins, CO

  INTEGRATED MEP - Mechanical, Electrical & Plumbing Engineering
      Thomas Segelhorst, PE, LEED AP | Lawrence Smith, PE | Fort Collins, CO

  JENSEN HUGHES - Fire Safety & Code Consulting (As-Needed)
      David Wolf, PE

  DFH CONSULTING - Cost Estimation (As-Needed)
      David Hoffman, PE

NOTABLE PROJECTS (COLLAB ARCHITECTURE):
  - City of Aurora, Municipal Center Space Planning & Remodel - Aurora, CO (289,000 sf)
  - Town of Superior, Downtown Civic Space - Superior, CO
  - Adams County, Western Service Center 3rd Floor Programming - Westminster, CO
  - Broomfield Police Department, Evidence Storage Facility - Broomfield, CO
  - Adams County, Honnen Facility Conditions Assessment - Brighton, CO
  - Department of Public Safety, Admin & Training Facility - Windsor, CO
  - Eaton Public Library, Renovation & Addition - Eaton, CO
  - Town of Silverthorne, Recreation Center Expansion - Silverthorne, CO
  - Town of Estes Park, Transit Facility - Estes Park, CO
  - Weld County, Grounds Building Design - Greeley, CO

NOTABLE PROJECTS (TEAM - PREVIOUS FIRMS):
  - Town of Timnath, Police Services Building, 29,000 sf (Kala Bailor) - Timnath, CO
  - Town of Timnath, Town Center, 15,250 sf (Kala Bailor) - Timnath, CO
  - Town of Windsor, Public Works Campus, 51,500 sf (Jordan Lockner) - Windsor, CO
  - Larimer County Police and Courts Addition (Blake Larsen) - Loveland, CO
  - City of Loveland, Fire Station 3 (Thomas Segelhorst) - Loveland, CO
  - City of Loveland, Fire Station 4 (Thomas Segelhorst) - Loveland, CO

BILLING RATES:
  - Principal Architect / Engineer:  $225/hr
  - Project Architect / Engineer:    $205/hr
  - Project Manager / Engineer:      $185/hr
  - QA/QC Review:                    $185/hr
  - CAD Technician:                  $115/hr
  - Interior Designer:                $95/hr
  - Administrative:                   $75/hr

REIMBURSABLES:
  - Outside Materials / Services / Supplies:  Cost + 15%
  - Mileage:                                  $0.70 / mile

REFERENCES:
  - Robert Wynkoop, Police Sergeant, Town of Timnath
      970.224.3211 | rwynkoop@timnathgov.com
  - Brian Rowe, Deputy Director of Public Works, Town of Windsor
      970.674.5400 | browe@windsorgov.com
  - Elly Watson, Business Services Manager, City of Aurora
      303.739.7109 | elwatson@auroragov.org
  - Jordan Hayes, Parks & Recreation Analyst II, Town of Superior
      303.499.3675 | jordanh@superiorcolorado.gov
  - Kyle Burg, Project Manager, Facilities & Fleet Mgmt., Adams County
      720.523.6062 | KBurg@adcogov.org
"""
print("Firm information loaded.")


Firm information loaded.


## Step 1 — Extract PDF Text

Pull text from the one-shot example proposal and the target RFQ.  
Run these two cells before any generation cell.


In [4]:
# ── Extract One-Shot Example Proposal ─────────────────────────────────────────
def extract_pdf(path):
    """Extract all text from a PDF, page-labelled."""
    pages = []
    with pdfplumber.open(str(path)) as pdf:
        for i, page in enumerate(pdf.pages):
            text = page.extract_text()
            if text and text.strip():
                pages.append(f"[Page {i+1}]\n{text.strip()}")
    return "\n\n".join(pages)

example_text = extract_pdf(EXAMPLE_PDF)
print(f"Example proposal: {len(example_text):,} chars across {example_text.count('[Page')} pages")
print("\nFirst 500 chars:")
print(example_text[:500], "...")


Example proposal: 41,725 chars across 19 pages

First 500 chars:
[Page 1]
RESPONSE TO RFP
RFP #2025-22
CITY OF LOVELAND
DESIGN SERVICES FOR POLICE AND
COURTS BUILDING RENOVATION
MARCH 20, 2025
970-292-7078 | WWW.COLLABARCHITECTS.COM | 9217 EASTMAN PARK DR. WINDSOR, CO 80550

[Page 2]
CITY OF LOVELAND
DESIGN SERVICES FOR POLICE AND COURTS BUILDING RENOVATION
RFP #2025-22
TABLE OF
CONTENTS
A./ COVER LETTER...........................................................................................3
B./ RELEVANT PROJECT EXPERIENCE.................................. ...


In [5]:
# ── Extract Target RFQ PDF ─────────────────────────────────────────────────────
# Re-run this cell alone when you swap in a new RFQ PDF.

rfq_text = extract_pdf(RFQ_PDF)
print(f"Target RFQ: {len(rfq_text):,} chars across {rfq_text.count('[Page')} pages")
print("\nFirst 500 chars:")
print(rfq_text[:500], "...")


Target RFQ: 36,978 chars across 17 pages

First 500 chars:
[Page 1]
REQUEST FOR QUALIFICATIONS
Architectural and Engineering Services
For
Cascade Campus Facility Renovation
Bid No. 2025-075
Owner:
City of Loveland
Public Works Department/Facilities Division
105 West 5th Street
Loveland, CO 80537
Issue Date:
December 5, 2025

[Page 2]
REQUEST FOR QUALIFICATIONS
The City of Loveland, Colorado (“City”) is seeking Statements of Qualifications (SOQ) from
qualified architectural and engineering firms (“Consultant”) to provide architectural and
engineering ser ...


## Step 2 — Initialize Claude Client

Sets up the streaming generator with a cached multi-turn message structure:
- **Turn 1** (cached): example proposal → Claude learns the style
- **Turn 2** (cached): RFQ + firm info → context for this bid
- **Turn 3**: the specific section task


In [ ]:
# ── Claude Client + Cached Message Builder ────────────────────────────────────
import time

client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)

SYSTEM_PROMPT = """\
You are a senior proposal writer for a professional architecture firm.
Generate compelling, client-focused proposal content from the RFQ and firm information.

RULES (non-negotiable):
- Output ONLY valid JSON — no markdown fences, no commentary outside the JSON object.
- Write in first-person plural: "our team", "we will", "we bring".
- Reference specific RFQ details: project name, scope, evaluation criteria, constraints.
- Name real team members and real past projects from the firm information provided.
- Match the professional tone of the example proposal you were shown.
- Do not invent facts; if a detail is unavailable, note what should be inserted.
"""


def _messages(task_prompt):
    """
    Three-turn cached message structure.
    After the first API call, subsequent calls read from cache (very low cost).
    """
    return [
        # Turn 1: one-shot style example (cached)
        {
            "role": "user",
            "content": [{
                "type": "text",
                "text": (
                    "Study this completed proposal carefully — match its professional "
                    "voice, structure, and level of detail precisely.\n\n"
                    "## COMPLETED EXAMPLE PROPOSAL\n\n" + example_text
                ),
                "cache_control": {"type": "ephemeral"},
            }],
        },
        {
            "role": "assistant",
            "content": (
                "I have carefully reviewed the example proposal and understand "
                "its tone, structure, and style. Ready to write."
            ),
        },
        # Turn 2: target RFQ + firm info (cached) + task
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": f"## TARGET RFQ\n\n{rfq_text}\n\n## FIRM INFORMATION\n\n{FIRM_INFO}",
                    "cache_control": {"type": "ephemeral"},
                },
                {"type": "text", "text": task_prompt},
            ],
        },
    ]


def generate(label, prompt, max_tokens=2000, max_retries=5):
    """Stream one proposal section with exponential-backoff retry on overload errors."""
    print(f"\n{'=' * 60}\n  {label}\n{'=' * 60}\n")
    for attempt in range(1, max_retries + 1):
        result = ""
        try:
            with client.messages.stream(
                model="claude-opus-4-8",
                max_tokens=max_tokens,
                system=SYSTEM_PROMPT,
                messages=_messages(prompt),
            ) as stream:
                for chunk in stream.text_stream:
                    print(chunk, end="", flush=True)
                    result += chunk
                usage = stream.get_final_message().usage
            print(
                f"\n\n[tokens — in: {usage.input_tokens:,}  out: {usage.output_tokens:,}  "
                f"cache_read: {usage.cache_read_input_tokens:,}  "
                f"cache_write: {usage.cache_creation_input_tokens:,}]"
            )
            return result
        except Exception as _err:
            err_str = str(_err).lower()
            is_overload = "overload" in err_str or "529" in err_str or "rate" in err_str
            if is_overload and attempt < max_retries:
                wait = 2 ** attempt  # 2, 4, 8, 16 seconds
                print(f"\n  ⚠  API overloaded (attempt {attempt}/{max_retries}) — retrying in {wait}s...")
                time.sleep(wait)
            else:
                raise


def parse_json(raw):
    """Strip optional markdown fences and parse JSON."""
    clean = re.sub(r'^```[a-z]*\n?|\n?```$', '', raw.strip(), flags=re.MULTILINE)
    return json.loads(clean.strip())


print("Claude client ready. Model: claude-opus-4-8")
print("Run each section cell below independently — cached after the first call.")
print("Retry logic: up to 5 attempts with exponential backoff on overload errors.")


## Step 3 — Generate Proposal Sections

Each cell below calls Claude once and returns structured JSON.  
The RFQ and example proposal are cached after the first call — subsequent sections cost only output tokens.

Run cells in order, or re-run any single section to regenerate it.


In [7]:
# ── Section 1: Cover Letter ───────────────────────────────────────────────────
# Returns: rfp_title, rfp_number, client info, re_line, salutation, body (4 paragraphs)

_cover_raw = generate(
    "COVER LETTER",
    f"""
Extract RFQ metadata and write a 4-paragraph cover letter body.
Return a JSON object with EXACTLY these keys (no extras):

  rfp_title       — full project / RFQ title (string)
  rfp_number      — bid or RFQ number, e.g. "Bid 2025-075" (string)
  client_name     — issuing organization name (string)
  client_dept     — department name or "" (string)
  client_address1 — street address (string)
  client_address2 — city, state, zip (string)
  re_line         — full "Re:" line text (string)
  salutation      — e.g. "Dear Selection Committee," (string)
  body            — exactly 4 flowing prose paragraphs separated by \\n\\n:
                    Para 1: understanding of the project scope and the client's core need
                    Para 2: why Collab Architecture is uniquely qualified (name 1-2 past projects)
                    Para 3: introduce Jordan Lockner (Principal), Kala Bailor (PM), Michael Aller (QA/QC)
                    Para 4: closing commitment and call to action (2 sentences)

Today is {date.today().strftime("%B %d, %Y")}.
Output ONLY valid JSON with no markdown fences.
""",
    max_tokens=2000,
)

cover = parse_json(_cover_raw)
print("\n--- Parsed cover letter fields ---")
for k, v in cover.items():
    if k != "body":
        print(f"  {k:18s}: {str(v)[:80]}")
print(f"  {'body':18s}: {str(cover.get('body',''))[:100]}...")



  COVER LETTER

{
  "rfp_title": "Architectural and Engineering Services for Cascade Campus Facility Renovation",
  "rfp_number": "Bid No. 2025-075",
  "client_name": "City of Loveland",
  "client_dept": "Public Works Department / Facilities Division",
  "client_address1": "105 West 5th Street",
  "client_address2": "Loveland, CO 80537",
  "re_line": "Re: Bid No. 2025-075 — Architectural and Engineering Services for Cascade Campus Facility Renovation",
  "salutation": "To the City of Loveland Staff & Selection Committee,",
  "body": "After reviewing the City of Loveland's RFQ, attending the mandatory pre-submittal meeting at the Cascade Campus, and carefully evaluating the project's objectives, we recognize the significance of this renovation in supporting the long-term growth of the Loveland Utilities Department. The renovation of approximately 62,030 SF across the entire second floor and portions of the first floor of the 1515 Cascade Avenue facility represents a critical step in ac

In [8]:
# ── Section 2: Firm Profile ───────────────────────────────────────────────────
# Returns: headline, who_we_are (2-3 paragraphs), differentiator

_firm_raw = generate(
    "FIRM PROFILE",
    """
Write firm profile content tailored to this specific RFQ scope.
Return a JSON object with EXACTLY these keys:

  headline       — short display tagline (10 words max, no period) for the page header (string)
  who_we_are     — 2-3 flowing prose paragraphs (joined by \\n\\n) for the "WHO WE ARE"
                   section: introduce Collab Architecture, its founding year, philosophy,
                   growth recognition, and municipal/civic project expertise (string)
  differentiator — 1 focused prose paragraph on why Collab is uniquely right for
                   THIS specific project and client (string)

Output ONLY valid JSON with no markdown fences.
""",
    max_tokens=2000,
)

firm = parse_json(_firm_raw)
print("\n--- Parsed firm profile ---")
print(f"  Headline:       {firm.get('headline','')}")
print(f"  Who we are:     {str(firm.get('who_we_are',''))[:120]}...")
print(f"  Differentiator: {str(firm.get('differentiator',''))[:100]}...")



  FIRM PROFILE

{
  "headline": "Renovating Loveland's Cascade Campus for the Utilities Department's Future",
  "who_we_are": "Collab Architecture is a Windsor-based architecture and interior design firm founded in 2020 and dedicated to bringing the power of collaborative design to create a stronger, better, and more sustainable community. Our holistic approach to every project begins with two fundamental questions: What design problem are we looking to solve, and how can we make the project the most successful for our client? It all starts with listening. \"Stop. Collaborate and Listen\" is our unofficial motto, and we push to live by those words on every engagement. This collaborative approach has proven successful and has helped us mitigate preventable issues early in the design process—an essential discipline for a phased renovation like the Cascade Campus, where occupied tenant and staff areas must remain operational throughout design and construction.\n\nIn both 2023 and 2024, C

In [9]:
# ── Section 3: Team Bios ──────────────────────────────────────────────────────
# Returns bios for Jordan Lockner, Kala Bailor, and Michael Aller.

_team_raw = generate(
    "TEAM BIOS",
    """
Write bio content for three key team members.
Return a JSON object with keys "lockner", "bailor", and "aller".
Each value is an object with EXACTLY these keys:

  name          — full name with credentials, e.g. "Jordan W. Lockner, AIA, NCARB" (string)
  title         — job title on THIS project (string)
  role_label    — role label line, e.g. "Project Role: Principal Architect / Primary Contact" (string)
  bio           — 3-4 sentence prose paragraph on their expertise and fit for this RFQ (string)
  education     — degree and institution, e.g. "University of Colorado, B.ENVD" (string)
  registrations — comma-separated licenses/credentials (string)
  experience    — JSON array of 6-8 strings, each "Client, Project Name - City, ST"
                  (pull from firm's documented project history; note "(previous firm)" where applicable)

Use ONLY facts from the firm information. Tailor each bio to this RFQ's scope.
Output ONLY valid JSON with no markdown fences.
""",
    max_tokens=2500,
)

team = parse_json(_team_raw)
print("\n--- Parsed team bios ---")
for key in ["lockner", "bailor", "aller"]:
    m = team.get(key, {})
    print(f"  {key:10s}: {m.get('name','')}  |  {m.get('title','')}")
    print(f"             {len(m.get('experience',[]))} experience entries")



  TEAM BIOS

{
  "lockner": {
    "name": "Jordan W. Lockner, AIA, NCARB",
    "title": "Principal Architect & Owner",
    "role_label": "Project Role: Principal Architect / Primary Contact",
    "bio": "Jordan believes that the root of all good architecture must stem from and support the community that it exists within, an ethos that directly aligns with the City of Loveland's goal of expanding the Utilities Department's Cascade Campus to serve long-term staffing growth. His ability to listen, understand, and collaborate with stakeholders while coordinating the full design process has produced successful outcomes on municipal renovations and space-planning efforts, including his role as Project Manager on the 51,500 SF Town of Windsor Public Works Campus. He is well versed in public projects throughout Northern Colorado, with deep familiarity in local codes, permitting, and the design-bid-build delivery method anticipated for this renovation. Jordan has been recognized for his leader

In [10]:
# ── Section 4: Relevant Project Experience ────────────────────────────────────
# Returns 2 project case studies most relevant to this RFQ.

_exp_raw = generate(
    "RELEVANT PROJECT EXPERIENCE",
    """
Select and write 2 project case studies most directly relevant to this RFQ.
Return a JSON object with keys "project1" and "project2". Each value has EXACTLY:

  title         — project name in UPPERCASE (string)
  location      — "City, ST" (string)
  team_lead     — Collab team member who led it (string)
  project_type  — e.g. "Interior Renovation, Space Planning" (string)
  services      — e.g. "Architecture, Interior Design, Construction Administration" (string)
  description   — 2-3 flowing prose paragraphs (joined by \\n\\n):
                  what was designed/built, why it is relevant to THIS RFQ scope,
                  one concrete outcome or measurable result

Select ONLY from the firm's documented past projects in the firm information.
Output ONLY valid JSON with no markdown fences.
""",
    max_tokens=1500,
)

experience = parse_json(_exp_raw)
print("\n--- Parsed project experience ---")
for key in ["project1", "project2"]:
    p = experience.get(key, {})
    print(f"  {key}: {p.get('title','')[:65]}")
    print(f"         {p.get('location','')}  |  Lead: {p.get('team_lead','')}")



  RELEVANT PROJECT EXPERIENCE

{
  "project1": {
    "title": "CITY OF AURORA, MUNICIPAL CENTER SPACE PLANNING & REMODEL",
    "location": "Aurora, CO",
    "team_lead": "Michael Aller, AIA, LEED AP",
    "project_type": "Multi-Story Interior Renovation, Space Planning & Programming",
    "services": "Architecture, Interior Design, Space Planning, Programming, Construction Administration",
    "description": "Collab Architecture began its partnership with the City of Aurora through a comprehensive space planning effort and audit of the 289,000-square-foot Aurora Municipal Center (AMC). This project involved evaluating space utilization across five stories and 20 municipal departments to address the facility's full capacity and prepare for future growth. By assessing operational needs, hybrid work opportunities, and potential department consolidations, our team developed a road-map to enhance efficiency, streamline workflows, and support scalability within the AMC — work directly paral

In [11]:
# ── Section 5: Project Understanding & Approach ───────────────────────────────
# Returns approach narrative, key challenges, schedule framing, and references.

_approach_raw = generate(
    "PROJECT UNDERSTANDING & APPROACH",
    """
Write project understanding and approach content.
Return a JSON object with EXACTLY these keys:

  intro           — 2 flowing prose paragraphs (joined by \\n\\n) showing deep understanding
                    of this RFQ: reference specific scope elements, constraints, evaluation
                    criteria, and stated client goals from the RFQ document (string)
  key_challenges  — 2-3 sentences identifying the single biggest project challenge
                    and our specific mitigation strategy (string)
  schedule_intro  — 1-2 sentences framing our proposed schedule relative to the RFQ's timeline (string)
  references      — JSON array of exactly 3 strings, each formatted as:
                    "Name, Title, Organization\\nPhone  |  Email"
                    Use actual references from the firm information.

Output ONLY valid JSON with no markdown fences.
""",
    max_tokens=1800,
)

approach = parse_json(_approach_raw)
print("\n--- Parsed approach ---")
print(f"  Intro:           {str(approach.get('intro',''))[:100]}...")
print(f"  Key challenges:  {str(approach.get('key_challenges',''))[:100]}...")
print(f"  References:      {len(approach.get('references', []))} entries")
for ref in approach.get('references', []):
    print(f"    - {ref.split(chr(10))[0][:70]}")



  PROJECT UNDERSTANDING & APPROACH

{
  "intro": "The City of Loveland's acquisition of the Cascade Campus represents a forward-thinking investment in the long-term growth of the Loveland Utilities Department, and we recognize that this renovation must thoughtfully balance immediate programmatic needs with the 15- and 30-year staffing projections that informed the City's 2023 programming efforts. Our team understands the full breadth of the scope outlined in the RFQ: the renovation of approximately 62,030 SF across the entire second floor and portions of the first floor, encompassing interior finish updates, space planning to City standards, collaborative teaming and conference spaces with audio/visual improvements, break areas and outdoor amenities, comprehensive MEP and network upgrades, ADA compliance improvements, and exterior site enhancements including parking, lighting, EV charging, and security. We are equally prepared to address the project's defining constraint—the continued

In [12]:
# ── Section 6: Fee Schedule ────────────────────────────────────────────────────
# Returns billing rates, reimbursables, and fee narrative.

_fee_raw = generate(
    "FEE SCHEDULE",
    """
Write fee schedule content.
Return a JSON object with EXACTLY these keys:

  narrative      — 2-3 sentences describing our fee approach: unit-price NTE basis,
                   phase-level budget caps, monthly earned-value reporting (string)
  rates          — JSON array of objects {"role": "...", "rate": "$X / hour"}
                   Use the ACTUAL billing rates from firm information (array)
  reimbursables  — JSON array of objects {"item": "...", "basis": "..."}
                   Use the ACTUAL reimbursable policy from firm information (array)

Output ONLY valid JSON with no markdown fences.
""",
    max_tokens=1000,
)

fee = parse_json(_fee_raw)
print("\n--- Parsed fee schedule ---")
print(f"  Narrative:  {str(fee.get('narrative',''))[:100]}...")
print(f"  Rates ({len(fee.get('rates',[]))} roles):")
for r in fee.get('rates', []):
    print(f"    {r.get('role',''):35s}  {r.get('rate','')}")
print(f"  Reimbursables ({len(fee.get('reimbursables',[]))} items):")
for r in fee.get('reimbursables', []):
    print(f"    {r.get('item',''):35s}  {r.get('basis','')}")



  FEE SCHEDULE

{
  "narrative": "Our fee for the Cascade Campus Facility Renovation is structured on a unit-price, not-to-exceed (NTE) basis aligned with the City of Loveland's purchasing requirements and the Design-Bid-Build delivery method outlined in the RFQ. We establish phase-level budget caps spanning Conceptual Design and Programming, Schematic Design, Design Development, Construction Documents, Bidding and Procurement, Construction Administration, and Project Closeout, ensuring transparent cost control across the entire 62,030 SF scope. Our team provides monthly earned-value reporting that tracks completed work against each phase budget, giving Facilities and Loveland Utilities Department stakeholders clear visibility into progress, scope, and remaining contract value throughout the project.",
  "rates": [
    { "role": "Principal Architect / Engineer", "rate": "$225 / hour" },
    { "role": "Project Architect / Engineer", "rate": "$205 / hour" },
    { "role": "Project Manag

In [ ]:
# ── Section 7: Proposed Project Schedule ─────────────────────────────────────────────
# Uses one-shot learning from the example proposal's schedule structure
# to project phases, tasks, durations, and calendar months for this RFQ.

_schedule_raw = generate(
    "PROPOSED PROJECT SCHEDULE",
    f"""
Write a proposed project schedule tailored to this RFQ's scope and stated timeline.
Use the example proposal's schedule as a structural guide for phase naming,
task granularity, and duration estimates.
Return a JSON object with EXACTLY these keys (no extras):

  intro               — 1-2 sentence paragraph introducing the schedule, naming the
                        client/entity, type of services, and anticipated completion
                        timeframe drawn from the RFQ. Write in first-person plural.
                        (string)

  project_short_title — concise project title for the schedule calendar header,
                        max 60 characters (string)

  phases              — array of exactly 4 objects, each with:
                          title    — "Phase N: [Phase Name]" e.g.
                                     "Phase 1: Project Initiation" (string)
                          duration — estimated weeks, e.g. "3 Weeks" or "4 Weeks";
                                     use "Future Phase" for the final phase if it
                                     covers bidding, permitting, or CA (string)
                          tasks    — 4–8 concise deliverable / task descriptions
                                     drawn from the RFQ scope (array of strings)

  calendar_months     — array of exactly 4 full month names matching the expected
                        project start and duration, e.g.
                        ["September", "October", "November", "December"] (array)

  calendar_year       — 4-digit year string for the schedule, e.g. "2025" or "2026"
"""
)

schedule = json.loads(_schedule_raw)

display(Markdown(
    f"**Proposed Schedule — {schedule['project_short_title']}**\n\n"
    + schedule['intro'] + "\n\n"
    + "\n\n".join(
        f"**{p['title']}** ({p['duration']})\n"
        + "\n".join(f"- {t}" for t in p['tasks'])
        for p in schedule['phases']
    )
    + f"\n\n*Calendar: {', '.join(schedule['calendar_months'])} {schedule['calendar_year']}*"
))


## Step 4 — IDML Template Injection

The three cells below do not call Claude — they build XML and write files.

1. **Helper utilities** — XML escaping, Markdown stripping, story envelope
2. **Story builders** — One function per template section type
3. **Build IDML** — Applies all story updates and writes the modified `.idml` file


In [13]:
# ── IDML XML Helper Utilities ─────────────────────────────────────────────────

def xe(text):
    """XML-escape a string for IDML Content elements."""
    return html.escape(str(text), quote=False)

def strip_md(text):
    """Remove Markdown syntax, returning clean plain text."""
    text = re.sub(r'^#{1,6}\s+', '', text, flags=re.MULTILINE)
    text = re.sub(r'\*\*([^*]+)\*\*', r'\1', text)
    text = re.sub(r'\*([^*]+)\*', r'\1', text)
    text = re.sub(r'^\s*[-*]\s+', '', text, flags=re.MULTILINE)
    text = re.sub(r'^---+$', '', text, flags=re.MULTILINE)
    text = re.sub(r'\|[^\n]*\|[^\n]*\n', '', text)
    text = re.sub(r'\[([^\]]+)\]\([^)]+\)', r'\1', text)
    text = re.sub(r'\n{3,}', '\n\n', text)
    return text.strip()

def wrap_story(self_id, inner_xml):
    """Wrap inner paragraph XML in the standard IDML <Story> envelope."""
    return (
        '<?xml version="1.0" encoding="UTF-8" standalone="yes"?>\n'
        '<idPkg:Story xmlns:idPkg="http://ns.adobe.com/AdobeInDesign/idml/1.0/packaging"'
        ' DOMVersion="21.2">\n'
        f'\t<Story Self="{self_id}" UserText="true" IsEndnoteStory="false"'
        ' AppliedTOCStyle="n" TrackChanges="false" StoryTitle="$ID/" AppliedNamedGrid="n">\n'
        '\t\t<StoryPreference OpticalMarginAlignment="false" OpticalMarginSize="12"'
        ' FrameType="TextFrameType" StoryOrientation="Horizontal"'
        ' StoryDirection="LeftToRightDirection" />\n'
        '\t\t<InCopyExportOption IncludeGraphicProxies="true"'
        ' IncludeAllResources="false" />\n'
        f'{inner_xml}\n'
        '\t</Story>\n'
        '</idPkg:Story>'
    )

def pb(text, style="Body", char_style="$ID/[No character style]", br=True):
    """Build one ParagraphStyleRange / CharacterStyleRange block."""
    br_tag = '\n\t\t\t\t<Br />' if br else ''
    return (
        f'\t\t<ParagraphStyleRange AppliedParagraphStyle="ParagraphStyle/{xe(style)}">\n'
        f'\t\t\t<CharacterStyleRange AppliedCharacterStyle="CharacterStyle/{xe(char_style)}">\n'
        f'\t\t\t\t<Content>{xe(text)}</Content>{br_tag}\n'
        f'\t\t\t</CharacterStyleRange>\n'
        f'\t\t</ParagraphStyleRange>'
    )

def pbs(text, style="Body"):
    """Convert multi-paragraph plain text into consecutive paragraph blocks."""
    chunks = [c.strip() for c in text.split('\n\n') if c.strip()]
    return '\n'.join(pb(c, style) for c in chunks)

def rc(xml, old, new):
    """Replace the first <Content>old</Content> in a story XML string."""
    return xml.replace(
        f'<Content>{html.escape(old, quote=False)}</Content>',
        f'<Content>{xe(new)}</Content>',
        1,
    )

print("IDML helper utilities loaded (xe, strip_md, wrap_story, pb, pbs, rc).")


IDML helper utilities loaded (xe, strip_md, wrap_story, pb, pbs, rc).


In [ ]:
# ── Story XML Construction Functions ─────────────────────────────────────────
# Each function takes parsed JSON data and returns a complete IDML Story XML string.

def story_cover_letter(c):
    """Story u221 — full cover letter body with recipient block and signature."""
    body_chunks = [ch.strip() for ch in c['body'].split('\n\n') if ch.strip()]
    content = [
        f'\t\t\t\t<Content>{xe(c["salutation"])}</Content>\n\t\t\t\t<Br />',
        '\t\t\t\t<Br />',
    ]
    for ch in body_chunks:
        content += [f'\t\t\t\t<Content>{xe(ch)}</Content>', '\t\t\t\t<Br />', '\t\t\t\t<Br />']
    content += [
        '\t\t\t\t<Content>Sincerely,</Content>', '\t\t\t\t<Br />',
        '\t\t\t\t<Br />', '\t\t\t\t<Br />',
        '\t\t\t\t<Content>Jordan W. Lockner, AIA, NCARB</Content>', '\t\t\t\t<Br />',
        '\t\t\t\t<Content>Founding Principal  |  Collab Architecture</Content>', '\t\t\t\t<Br />',
        '\t\t\t\t<Content>jordan@collabarchitects.com  |  970.215.9907</Content>',
    ]
    inner = (
        '\t\t<ParagraphStyleRange AppliedParagraphStyle="ParagraphStyle/Body - no spacing">\n'
        '\t\t\t<CharacterStyleRange AppliedCharacterStyle="CharacterStyle/$ID/[No character style]">\n'
        f'\t\t\t\t<Content>{xe(c["client_name"])}</Content>\n\t\t\t\t<Br />\n'
        f'\t\t\t\t<Content>{xe(c.get("client_dept") or c["client_name"])}</Content>\n\t\t\t\t<Br />\n'
        f'\t\t\t\t<Content>{xe(c["client_address1"])}</Content>\n\t\t\t\t<Br />\n'
        f'\t\t\t\t<Content>{xe(c["client_address2"])}</Content>\n\t\t\t\t<Br />\n'
        '\t\t\t</CharacterStyleRange>\n\t\t</ParagraphStyleRange>\n'
        '\t\t<ParagraphStyleRange AppliedParagraphStyle="ParagraphStyle/Body">\n'
        '\t\t\t<CharacterStyleRange AppliedCharacterStyle="CharacterStyle/$ID/[No character style]">\n'
        '\t\t\t\t<Br />\n\t\t\t</CharacterStyleRange>\n'
        '\t\t\t<CharacterStyleRange AppliedCharacterStyle="CharacterStyle/Bold, Teal">\n'
        f'\t\t\t\t<Content>{xe(c["re_line"])}</Content>\n\t\t\t\t<Br />\n'
        '\t\t\t</CharacterStyleRange>\n'
        '\t\t\t<CharacterStyleRange AppliedCharacterStyle="CharacterStyle/$ID/[No character style]">\n'
        + '\n'.join(content) + '\n'
        '\t\t\t</CharacterStyleRange>\n\t\t</ParagraphStyleRange>'
    )
    return wrap_story('u221', inner)


def story_team_name(sid, m):
    """Team name + title stories: u67f (Jordan), u703 (Kala), u918 (Mick)."""
    inner = (
        '\t\t<ParagraphStyleRange AppliedParagraphStyle="ParagraphStyle/Resumes:Name &amp; Credentials">\n'
        '\t\t\t<CharacterStyleRange AppliedCharacterStyle="CharacterStyle/$ID/[No character style]">\n'
        f'\t\t\t\t<Content>{xe(m["name"])}</Content>\n\t\t\t\t<Br />\n'
        '\t\t\t</CharacterStyleRange>\n\t\t</ParagraphStyleRange>\n'
        '\t\t<ParagraphStyleRange AppliedParagraphStyle="ParagraphStyle/Resumes:Job Title">\n'
        '\t\t\t<CharacterStyleRange AppliedCharacterStyle="CharacterStyle/$ID/[No character style]">\n'
        f'\t\t\t\t<Content>{xe(m["title"])}</Content>\n'
        '\t\t\t</CharacterStyleRange>\n\t\t</ParagraphStyleRange>'
    )
    return wrap_story(sid, inner)


def story_team_bio(sid, m):
    """Team role + bio stories: u665 (Jordan), u6ea (Kala), u8b3 (Mick)."""
    inner = (
        '\t\t<ParagraphStyleRange AppliedParagraphStyle="ParagraphStyle/Resumes:Resume Headers">\n'
        '\t\t\t<CharacterStyleRange AppliedCharacterStyle="CharacterStyle/$ID/[No character style]">\n'
        f'\t\t\t\t<Content>{xe(m["role_label"])}</Content>\n\t\t\t\t<Br />\n'
        '\t\t\t</CharacterStyleRange>\n\t\t</ParagraphStyleRange>\n'
        '\t\t<ParagraphStyleRange AppliedParagraphStyle="ParagraphStyle/Resumes:Resume Body">\n'
        '\t\t\t<CharacterStyleRange AppliedCharacterStyle="CharacterStyle/$ID/[No character style]">\n'
        f'\t\t\t\t<Content>{xe(strip_md(m["bio"]))}</Content>\n'
        '\t\t\t</CharacterStyleRange>\n\t\t</ParagraphStyleRange>'
    )
    return wrap_story(sid, inner)


def story_team_edu(sid, m):
    """Team education + registrations stories: u64c (Jordan), u6d0 (Kala), u84e (Mick)."""
    inner = (
        '\t\t<ParagraphStyleRange AppliedParagraphStyle="ParagraphStyle/Resumes:Resume Headers">\n'
        '\t\t\t<CharacterStyleRange AppliedCharacterStyle="CharacterStyle/Resumes:Resume Headers">\n'
        '\t\t\t\t<Content>Education</Content>\n\t\t\t\t<Br />\n'
        '\t\t\t</CharacterStyleRange>\n\t\t</ParagraphStyleRange>\n'
        '\t\t<ParagraphStyleRange AppliedParagraphStyle="ParagraphStyle/Resumes:Resume Body">\n'
        '\t\t\t<CharacterStyleRange AppliedCharacterStyle="CharacterStyle/$ID/[No character style]">\n'
        f'\t\t\t\t<Content>{xe(m["education"])}</Content>\n\t\t\t\t<Br />\n\t\t\t\t<Br />\n'
        '\t\t\t</CharacterStyleRange>\n'
        '\t\t\t<CharacterStyleRange AppliedCharacterStyle="CharacterStyle/Resumes:Resume Headers">\n'
        '\t\t\t\t<Content>Registrations &amp; Affiliations</Content>\n\t\t\t\t<Br />\n'
        '\t\t\t</CharacterStyleRange>\n'
        '\t\t\t<CharacterStyleRange AppliedCharacterStyle="CharacterStyle/$ID/[No character style]">\n'
        f'\t\t\t\t<Content>{xe(m["registrations"])}</Content>\n'
        '\t\t\t</CharacterStyleRange>\n\t\t</ParagraphStyleRange>'
    )
    return wrap_story(sid, inner)


def story_team_exp(sid, m):
    """Team select experience stories: u69e (Jordan), u71f (Kala), u986 (Mick)."""
    exp_lines = '\n'.join(
        f'\t\t\t\t<Content>{xe(p)}</Content>\n\t\t\t\t<Br />'
        for p in m.get('experience', [])
    )
    inner = (
        '\t\t<ParagraphStyleRange AppliedParagraphStyle="ParagraphStyle/Resumes:Resume Headers">\n'
        '\t\t\t<CharacterStyleRange AppliedCharacterStyle="CharacterStyle/Resumes:Resume Headers">\n'
        '\t\t\t\t<Content>Select Experience</Content>\n\t\t\t\t<Br />\n'
        '\t\t\t</CharacterStyleRange>\n\t\t</ParagraphStyleRange>\n'
        '\t\t<ParagraphStyleRange AppliedParagraphStyle="ParagraphStyle/Resumes:Resume Body">\n'
        '\t\t\t<CharacterStyleRange AppliedCharacterStyle="CharacterStyle/$ID/[No character style]">\n'
        f'{exp_lines}\n'
        '\t\t\t</CharacterStyleRange>\n\t\t</ParagraphStyleRange>'
    )
    return wrap_story(sid, inner)


def story_proj_title(sid, p):
    """Project title + location stories: u2ef (project 1), u3eb (project 2)."""
    inner = (
        '\t\t<ParagraphStyleRange AppliedParagraphStyle="ParagraphStyle/Experience:Project Name">\n'
        '\t\t\t<CharacterStyleRange AppliedCharacterStyle="CharacterStyle/$ID/[No character style]">\n'
        f'\t\t\t\t<Content>{xe(p["title"])}</Content>\n\t\t\t\t<Br />\n'
        '\t\t\t</CharacterStyleRange>\n\t\t</ParagraphStyleRange>\n'
        '\t\t<ParagraphStyleRange AppliedParagraphStyle="ParagraphStyle/Experience:City, State">\n'
        '\t\t\t<CharacterStyleRange AppliedCharacterStyle="CharacterStyle/$ID/[No character style]">\n'
        f'\t\t\t\t<Content>{xe(p["location"])}  |  Lead: {xe(p.get("team_lead",""))}</Content>\n'
        '\t\t\t</CharacterStyleRange>\n\t\t</ParagraphStyleRange>'
    )
    return wrap_story(sid, inner)


def story_proj_desc(sid, p):
    """Project description stories: u309 (project 1), u404 (project 2)."""
    chunks = [c.strip() for c in p['description'].split('\n\n') if c.strip()]
    blocks = '\n'.join(
        f'\t\t<ParagraphStyleRange AppliedParagraphStyle="ParagraphStyle/Body">\n'
        f'\t\t\t<CharacterStyleRange AppliedCharacterStyle="CharacterStyle/$ID/[No character style]">\n'
        f'\t\t\t\t<Content>{xe(c)}</Content>\n'
        f'\t\t\t</CharacterStyleRange>\n\t\t</ParagraphStyleRange>'
        for c in chunks
    )
    return wrap_story(sid, blocks)


def story_proj_meta(sid, p):
    """Project type + services stories: u322 (project 1), u41d (project 2)."""
    inner = (
        '\t\t<ParagraphStyleRange AppliedParagraphStyle="ParagraphStyle/Body - no spacing">\n'
        '\t\t\t<CharacterStyleRange AppliedCharacterStyle="CharacterStyle/$ID/[No character style]">\n'
        f'\t\t\t\t<Content>Type of Project: {xe(p.get("project_type",""))}</Content>\n\t\t\t\t<Br />\n'
        f'\t\t\t\t<Content>Services Provided: {xe(p.get("services",""))}</Content>\n'
        '\t\t\t</CharacterStyleRange>\n\t\t</ParagraphStyleRange>'
    )
    return wrap_story(sid, inner)


def story_references(refs):
    """References story u1552."""
    lines = []
    for ref in refs:
        for line in ref.split('\n'):
            if line.strip():
                lines.append(f'\t\t\t\t<Content>{xe(line.strip())}</Content>\n\t\t\t\t<Br />')
        lines.append('\t\t\t\t<Br />')
    inner = (
        '\t\t<ParagraphStyleRange AppliedParagraphStyle="ParagraphStyle/Body">\n'
        '\t\t\t<CharacterStyleRange AppliedCharacterStyle="CharacterStyle/$ID/[No character style]">\n'
        + '\n'.join(lines) + '\n'
        '\t\t\t</CharacterStyleRange>\n\t\t</ParagraphStyleRange>'
    )
    return wrap_story('u1552', inner)


def story_billing_rates(fee):
    """Standard hourly rates story u1d57."""
    rate_lines = []
    for r in fee.get('rates', []):
        rate_lines += [
            f'\t\t\t\t<Content>{xe(r.get("role",""))}</Content>\n\t\t\t\t<Br />',
            f'\t\t\t\t<Content>{xe(r.get("rate",""))}</Content>\n\t\t\t\t<Br />',
        ]
    inner = (
        '\t\t<ParagraphStyleRange AppliedParagraphStyle="ParagraphStyle/Body - no spacing">\n'
        '\t\t\t<CharacterStyleRange AppliedCharacterStyle="CharacterStyle/Bold, Teal">\n'
        '\t\t\t\t<Content>STANDARD HOURLY RATES</Content>\n\t\t\t\t<Br />\n'
        '\t\t\t</CharacterStyleRange>\n'
        '\t\t\t<CharacterStyleRange AppliedCharacterStyle="CharacterStyle/$ID/[No character style]">\n'
        + '\n'.join(rate_lines) + '\n'
        '\t\t\t</CharacterStyleRange>\n\t\t</ParagraphStyleRange>'
    )
    return wrap_story('u1d57', inner)


print("All story builder functions loaded.")
print("Builders: story_cover_letter, story_team_name/bio/edu/exp, story_proj_title/desc/meta,")
print("          story_references, story_billing_rates")


def story_schedule_intro(sched):
    """Proposed schedule intro paragraph — Story u17b6."""
    inner = (
        '\t\t<ParagraphStyleRange AppliedParagraphStyle="ParagraphStyle/Body">\n'
        '\t\t\t<CharacterStyleRange AppliedCharacterStyle="CharacterStyle/$ID/[No character style]">\n'
        f'\t\t\t\t<Content>{xe(sched["intro"])}</Content>\n'
        '\t\t\t</CharacterStyleRange>\n\t\t</ParagraphStyleRange>'
    )
    return wrap_story('u17b6', inner)


def story_schedule_phase(sid, phase):
    """Phase title + bulleted task list for the schedule Gantt — Stories u19a4/u19bd/u1a7c/u1a95."""
    tasks = phase.get('tasks', [])
    task_lines = []
    for i, task in enumerate(tasks):
        task_lines.append(f'\t\t\t\t<Content>{xe(task)}</Content>')
        if i < len(tasks) - 1:
            task_lines.append('\t\t\t\t<Br />')
    inner = (
        '\t\t<ParagraphStyleRange AppliedParagraphStyle="ParagraphStyle/Schedule%3aActivity Title">\n'
        '\t\t\t<CharacterStyleRange AppliedCharacterStyle="CharacterStyle/$ID/[No character style]">\n'
        f'\t\t\t\t<Content>{xe(phase["title"])}</Content>\n\t\t\t\t<Br />\n'
        '\t\t\t</CharacterStyleRange>\n\t\t</ParagraphStyleRange>\n'
        '\t\t<ParagraphStyleRange AppliedParagraphStyle="ParagraphStyle/Schedule%3aActivity List">\n'
        '\t\t\t<CharacterStyleRange AppliedCharacterStyle="CharacterStyle/Highlights">\n'
        + '\n'.join(task_lines) + '\n'
        '\t\t\t</CharacterStyleRange>\n\t\t</ParagraphStyleRange>'
    )
    return wrap_story(sid, inner)


def story_schedule_header(xml_str, sched):
    """Rewrite the schedule calendar header table in-place: project title and month columns."""
    months = sched.get('calendar_months', ['', '', '', ''])
    year   = sched.get('calendar_year', '2025')
    title  = sched.get('project_short_title', '')
    xml_str = xml_str.replace('insert RFP Title or project name', xe(title), 1)
    # Months 1 & 2 have a trailing space in the template Content node; 3 & 4 do not
    xml_str = xml_str.replace('>XX <', f'>{xe(months[0])} <', 1)
    xml_str = xml_str.replace('>XX <', f'>{xe(months[1])} <', 1)
    xml_str = xml_str.replace('>XX<', f'>{xe(months[2])}<', 1)
    xml_str = xml_str.replace('>XX<', f'>{xe(months[3])}<', 1)
    if year != '2025':
        xml_str = xml_str.replace('>2025<', f'>{xe(year)}<')
    return xml_str


print('Schedule story builders loaded: story_schedule_intro, story_schedule_phase, story_schedule_header')


## Step 5 — Build Modified IDML & Export PDF

Run these two cells after all generation cells have completed successfully.


In [ ]:
# ── Build Modified IDML ───────────────────────────────────────────────────────
# Maps every generated section to its IDML story file(s).
from urllib.parse import quote as _urlencode

_today = date.today().strftime("%B %d, %Y")

# ── Image link path rewriting ─────────────────────────────────────────────────
# The IDML template was authored on a different machine; its Spread files contain
# absolute LinkResourceURIs pointing to that machine's filesystem.  Rewrite them
# to the local Links folder so InDesign resolves images without manual relinking.
_LINKS_DIR     = _HERE / "1_RFP Base Template Files" / "Links"
_OLD_LINK_BASE = (
    "file:/Users/hollyfink/Desktop/2026%20Templates/"
    "2026%20Master%20Template_Facing%20Pages%20%5BLast%20Update%203.26.26%5D.idml/Links/"
)
_NEW_LINK_BASE = f"file:{_urlencode(str(_LINKS_DIR), safe='/')}/"

def rewrite_spread_links(xml_str):
    """Replace stale LinkResourceURI base with the local Links folder path."""
    return xml_str.replace(_OLD_LINK_BASE, _NEW_LINK_BASE)

STORY_MAP = {
    # ── Cover page fields ─────────────────────────────────────────────────────
    'Stories/Story_u1c6.xml':  lambda x: rc(x, 'RFP Title', cover['rfp_title']),
    'Stories/Story_u1ad.xml':  lambda x: rc(x, 'May 30, 2025', _today),
    'Stories/Story_u1df.xml':  lambda x: rc(x, 'Entity / client', cover['client_name']),
    'Stories/Story_u26c.xml':  lambda x: rc(x, 'Month Day, Year', _today),
    'Stories/Story_u2279.xml': lambda x: rc(x, 'RFP Title  |  Collab Architecture',
                                             f'{cover["rfp_title"]}  |  Collab Architecture'),
    'Stories/Story_u28b.xml':  lambda x: rc(x, 'RFP Title  |  Collab Architecture',
                                             f'{cover["rfp_title"]}  |  Collab Architecture'),

    # ── Cover letter ──────────────────────────────────────────────────────────
    'Stories/Story_u221.xml':  lambda x: story_cover_letter(cover),

    # ── Firm profile ──────────────────────────────────────────────────────────
    'Stories/Story_u1e69.xml': lambda x: rc(x,
        'Designing Spaces That Bring Communities Together', firm['headline']),
    'Stories/Story_u2054.xml': lambda x: rc(x,
        'Designing Spaces that Bring Communities Together.', firm['headline'] + '.'),
    'Stories/Story_u1e50.xml': lambda x: wrap_story('u1e50',
        pb('WHO WE ARE', 'Body', 'Bold, Teal') + '\n' + pbs(firm['who_we_are'], 'Body')),
    'Stories/Story_u2021.xml': lambda x: wrap_story('u2021',
        pbs(firm['who_we_are'] + '\n\n' + firm.get('differentiator', ''), 'Body')),
    'Stories/Story_u203b.xml': lambda x: wrap_story('u203b',
        pbs(approach['intro'], 'Body')),

    # ── Team: Jordan Lockner ──────────────────────────────────────────────────
    'Stories/Story_u67f.xml':  lambda x: story_team_name('u67f', team['lockner']),
    'Stories/Story_u665.xml':  lambda x: story_team_bio('u665',  team['lockner']),
    'Stories/Story_u64c.xml':  lambda x: story_team_edu('u64c',  team['lockner']),
    'Stories/Story_u69e.xml':  lambda x: story_team_exp('u69e',  team['lockner']),

    # ── Team: Kala Bailor ─────────────────────────────────────────────────────
    'Stories/Story_u703.xml':  lambda x: story_team_name('u703', team['bailor']),
    'Stories/Story_u6ea.xml':  lambda x: story_team_bio('u6ea',  team['bailor']),
    'Stories/Story_u6d0.xml':  lambda x: story_team_edu('u6d0',  team['bailor']),
    'Stories/Story_u71f.xml':  lambda x: story_team_exp('u71f',  team['bailor']),

    # ── Team: Michael Aller ───────────────────────────────────────────────────
    'Stories/Story_u918.xml':  lambda x: story_team_name('u918', team['aller']),
    'Stories/Story_u8b3.xml':  lambda x: story_team_bio('u8b3',  team['aller']),
    'Stories/Story_u84e.xml':  lambda x: story_team_edu('u84e',  team['aller']),
    'Stories/Story_u986.xml':  lambda x: story_team_exp('u986',  team['aller']),

    # ── Project experience 1 ──────────────────────────────────────────────────
    'Stories/Story_u2ef.xml':  lambda x: story_proj_title('u2ef', experience['project1']),
    'Stories/Story_u309.xml':  lambda x: story_proj_desc('u309',  experience['project1']),
    'Stories/Story_u322.xml':  lambda x: story_proj_meta('u322',  experience['project1']),

    # ── Project experience 2 ──────────────────────────────────────────────────
    'Stories/Story_u3eb.xml':  lambda x: story_proj_title('u3eb', experience['project2']),
    'Stories/Story_u404.xml':  lambda x: story_proj_desc('u404',  experience['project2']),
    'Stories/Story_u41d.xml':  lambda x: story_proj_meta('u41d',  experience['project2']),

    # ── References ────────────────────────────────────────────────────────────
    'Stories/Story_u1552.xml': lambda x: story_references(approach['references']),

    # ── Billing rates ─────────────────────────────────────────────────────────
    'Stories/Story_u1d57.xml': lambda x: story_billing_rates(fee),
    'Stories/Story_u1d70.xml': lambda x: rc(
        x,
        'Printing Services, Materials, SuppliesCost + 15%Mileage$.070 / mile',
        '  |  '.join(
            f'{r.get("item","")}: {r.get("basis","")}'
            for r in fee.get('reimbursables', [])
        ) or 'Outside Materials/Supplies: Cost + 15%  |  Mileage: $0.70/mile',
    ),

    # ── Proposed project schedule ──────────────────────────────────────────────
    'Stories/Story_u17b6.xml': lambda x: story_schedule_intro(schedule),
    'Stories/Story_u19a4.xml': lambda x: story_schedule_phase('u19a4', schedule['phases'][0]),
    'Stories/Story_u19bd.xml': lambda x: story_schedule_phase('u19bd', schedule['phases'][1]),
    'Stories/Story_u1a7c.xml': lambda x: story_schedule_phase('u1a7c', schedule['phases'][2]),
    'Stories/Story_u1a95.xml': lambda x: story_schedule_phase('u1a95', schedule['phases'][3]),
    'Stories/Story_u1b3a.xml': lambda x: rc(x, '3 Weeks',     schedule['phases'][0]['duration']),
    'Stories/Story_u19f5.xml': lambda x: rc(x, '4 Weeks',     schedule['phases'][1]['duration']),
    'Stories/Story_u1a46.xml': lambda x: rc(x, '5 Weeks',     schedule['phases'][2]['duration']),
    'Stories/Story_u1a2c.xml': lambda x: rc(x, 'Future Phase', schedule['phases'][3]['duration']),
    'Stories/Story_u1ace.xml': lambda x: story_schedule_header(x, schedule),
}

print(f"Applying {len(STORY_MAP)} story updates + image link rewriting...\n")
IDML_OUTPUT.parent.mkdir(parents=True, exist_ok=True)

_applied, _failed, _img_spreads = 0, 0, 0
with zipfile.ZipFile(str(IDML_TEMPLATE), 'r') as src:
    with zipfile.ZipFile(str(IDML_OUTPUT), 'w', zipfile.ZIP_DEFLATED) as dst:
        for item in src.infolist():
            raw = src.read(item.filename)
            if item.filename in STORY_MAP:
                try:
                    new_xml = STORY_MAP[item.filename](raw.decode('utf-8'))
                    dst.writestr(item, new_xml.encode('utf-8'))
                    print(f"  ✓  {item.filename.split('/')[-1]}")
                    _applied += 1
                except Exception as _e:
                    print(f"  ✗  {item.filename.split('/')[-1]}: {_e}")
                    dst.writestr(item, raw)
                    _failed += 1
            elif item.filename.startswith("Spreads/"):
                xml_str = raw.decode('utf-8')
                updated = rewrite_spread_links(xml_str)
                dst.writestr(item, updated.encode('utf-8'))
                if updated != xml_str:
                    _img_count = xml_str.count(_OLD_LINK_BASE)
                    print(f"  🖼  {item.filename.split('/')[-1]} ({_img_count} image link(s) updated)")
                    _img_spreads += 1
            else:
                dst.writestr(item, raw)

print(f"\n{'=' * 50}")
print(f"  Applied  : {_applied} stories  |  Errors: {_failed}  |  Image spreads: {_img_spreads}")
print(f"  IDML     → {IDML_OUTPUT}")
if _LINKS_DIR.exists():
    _img_files = [f for f in _LINKS_DIR.iterdir()
                  if f.suffix.lower() in ('.jpg', '.jpeg', '.png', '.tif', '.tiff')]
    print(f"  Links    → {_LINKS_DIR.name}/ ({len(_img_files)} image files ready)")
else:
    print(f"  ⚠  Links folder not found at {_LINKS_DIR}")

# ── Quick content verification ─────────────────────────────────────────────────
def _peek(story_file, max_chars=100):
    with zipfile.ZipFile(str(IDML_OUTPUT), 'r') as z:
        xml = z.read(story_file).decode('utf-8')
    return ' '.join(re.findall(r'<Content>([^<]+)</Content>', xml))[:max_chars]

print("\nSpot-checks:")
for label, sid in [
    ("RFP Title",     "Stories/Story_u1c6.xml"),
    ("Cover letter",  "Stories/Story_u221.xml"),
    ("Jordan name",   "Stories/Story_u67f.xml"),
    ("Project 1",     "Stories/Story_u2ef.xml"),
    ("Project 2",     "Stories/Story_u3eb.xml"),
    ("Billing rates", "Stories/Story_u1d57.xml"),
]:
    text = _peek(sid)
    ok = "✓" if text.strip() else "✗"
    print(f"  {ok}  {label:15s}: {text[:85]}")


In [16]:
# ── Export All Generated Text to Markdown ─────────────────────────────────────
# Saves the full proposal as generated_proposal.md in the same folder as this
# notebook, and renders a live preview below.

from IPython.display import display, Markdown as _MD
from pathlib import Path

_save_path = Path(r"/Users/brianpak/Desktop/Desktop - Brian’s MacBook Pro/Projects/Collab Architecture") / "generated_proposal.md"
_today_str = date.today().strftime("%B %d, %Y")

_sections = []

# Cover letter
_sections.append(f"""# {cover.get('rfp_title','RFP Response')}
**{cover.get('rfp_number','')}  |  Collab Architecture  |  {_today_str}**

---

## Cover Letter

{cover.get('client_name','')}  
{cover.get('client_dept','')}  
{cover.get('client_address1','')}  
{cover.get('client_address2','')}  

{cover.get('re_line','')}

{cover.get('salutation','')}

{cover.get('body','')}

Sincerely,

**Jordan W. Lockner, AIA, NCARB**  
Founding Principal | Collab Architecture  
jordan@collabarchitects.com | 970.215.9907""")

# Firm profile
_sections.append(f"""---

## Firm Profile

### {firm.get('headline','')}

{firm.get('who_we_are','')}

**Why Collab**

{firm.get('differentiator','')}""")

# Team bios
def _bio_block(key):
    m = team.get(key, {})
    exp = "\n".join(f"- {p}" for p in m.get('experience', []))
    return f"""### {m.get('name','')}
**{m.get('title','')}**

{m.get('bio','')}

*Education:* {m.get('education','')}  
*Registrations:* {m.get('registrations','')}

**Select Experience**

{exp}"""

_sections.append("---\n\n## Project Team\n\n" +
    "\n\n---\n\n".join(_bio_block(k) for k in ["lockner","bailor","aller"]))

# Project experience
def _proj_block(key):
    p = experience.get(key, {})
    return f"""### {p.get('title','')}
**{p.get('location','')}**  |  Lead: {p.get('team_lead','')}  
*Type:* {p.get('project_type','')}  
*Services:* {p.get('services','')}

{p.get('description','')}"""

_sections.append("---\n\n## Relevant Project Experience\n\n" +
    "\n\n---\n\n".join(_proj_block(k) for k in ["project1","project2"]))

# Approach + references
_ref_list = "\n\n".join(
    "**" + ref.split("\n")[0] + "**  \n" + (ref.split("\n")[1] if "\n" in ref else "")
    for ref in approach.get('references', [])
)
_sections.append(f"""---

## Project Understanding & Approach

{approach.get('intro','')}

**Key Challenge & Mitigation**

{approach.get('key_challenges','')}

**Schedule**

{approach.get('schedule_intro','')}

---

## References

{_ref_list}""")

# Fee schedule
_rates = "\n".join(f"| {r.get('role','')} | {r.get('rate','')} |" for r in fee.get('rates',[]))
_reimb = "\n".join(f"| {r.get('item','')} | {r.get('basis','')} |" for r in fee.get('reimbursables',[]))
_sections.append(f"""---

## Fee Schedule

{fee.get('narrative','')}

### Standard Hourly Rates

| Role | Rate |
|------|------|
{_rates}

### Reimbursable Expenses

| Item | Basis |
|------|-------|
{_reimb}""")

_full_md = "\n\n".join(_sections)
_save_path.write_text(_full_md, encoding="utf-8")
print(f"✓  Saved → {_save_path}")
print(f"   {len(_full_md):,} characters")
print()
display(_MD(_full_md))


✓  Saved → /Users/brianpak/Desktop/Desktop - Brian’s MacBook Pro/Projects/Collab Architecture/generated_proposal.md
   22,857 characters



# Architectural and Engineering Services for Cascade Campus Facility Renovation
**Bid No. 2025-075  |  Collab Architecture  |  June 09, 2026**

---

## Cover Letter

City of Loveland  
Public Works Department / Facilities Division  
105 West 5th Street  
Loveland, CO 80537  

Re: Bid No. 2025-075 — Architectural and Engineering Services for Cascade Campus Facility Renovation

To the City of Loveland Staff & Selection Committee,

After reviewing the City of Loveland's RFQ, attending the mandatory pre-submittal meeting at the Cascade Campus, and carefully evaluating the project's objectives, we recognize the significance of this renovation in supporting the long-term growth of the Loveland Utilities Department. The renovation of approximately 62,030 SF across the entire second floor and portions of the first floor of the 1515 Cascade Avenue facility represents a critical step in accommodating the Department's 15-year staffing projections while positioning the campus for future phases. Our team understands that success on this project hinges on thoughtful programming, careful space planning to City standards, modernization of interior finishes and MEP systems, ADA compliance improvements, and exterior site enhancements—all delivered while existing tenants and select City staff remain in the occupied building. We are prepared to develop a strict phasing and safety plan, in collaboration with the City's project team and selected general contractor, to ensure continuity of services throughout design and construction.

Collab Architecture, together with our trusted engineering partners, brings directly relevant experience in municipal renovations, large-scale space planning, and the integration of structural, mechanical, electrical, plumbing, and fire protection upgrades. Our work with the City of Aurora on the 289,000 SF Municipal Center Space Planning & Remodel—where we evaluated space utilization across five stories and twenty departments and reconfigured occupied administrative space—demonstrates our ability to optimize layouts for evolving operational needs within an active facility. Likewise, our ongoing partnership with Adams County on the Western Service Center 3rd Floor Programming reflects our expertise in ADA-compliant renovations, accessible restroom and lobby upgrades, and durable material selection for high-use public buildings. As a Northern Colorado firm that has completed over 30 projects in Loveland, we bring deep familiarity with the City's codes, permitting processes, and design standards.

I, Jordan Lockner, will serve as the Principal Architect and primary firm contact, providing strategic oversight and ensuring alignment with the City's goals and master planning efforts. Project Architect Kala Bailor will act as the City's Project Manager and primary day-to-day contact, leading programming analysis, stakeholder coordination, and space planning—building on her experience delivering the 29,000 SF Timnath Police Services Building and the Timnath Town Center. Michael Aller, with more than four decades of municipal and institutional design experience, will lead Quality Assurance and Quality Control, reviewing all documentation to ensure compliance with City standards and best practices. Our sub-consultant team—including Larsen Structural Design, whose principal Blake Larsen completed the Larimer County Police and Courts Addition in Loveland, and Integrated MEP, whose principal Thomas Segelhorst designed Loveland Fire Stations 3 and 4—ensures a seamless and proven design process tailored to the design-bid-build delivery method.

We are committed to delivering a well-coordinated, cost-conscious renovation that meets the Cascade Campus's programmatic needs while maintaining safe, uninterrupted operations for existing occupants. We appreciate your consideration and welcome the opportunity to discuss our approach with the City in greater detail.

Sincerely,

**Jordan W. Lockner, AIA, NCARB**  
Founding Principal | Collab Architecture  
jordan@collabarchitects.com | 970.215.9907

---

## Firm Profile

### Renovating Loveland's Cascade Campus for the Utilities Department's Future

Collab Architecture is a Windsor-based architecture and interior design firm founded in 2020 and dedicated to bringing the power of collaborative design to create a stronger, better, and more sustainable community. Our holistic approach to every project begins with two fundamental questions: What design problem are we looking to solve, and how can we make the project the most successful for our client? It all starts with listening. "Stop. Collaborate and Listen" is our unofficial motto, and we push to live by those words on every engagement. This collaborative approach has proven successful and has helped us mitigate preventable issues early in the design process—an essential discipline for a phased renovation like the Cascade Campus, where occupied tenant and staff areas must remain operational throughout design and construction.

In both 2023 and 2024, Collab Architecture was named one of the fastest-growing private companies in Northern Colorado and the Front Range. We provide full-service commercial architecture and interior design, with in-depth experience on projects ranging from tenant finishes and interior renovations to large ground-up structures. No two projects are the same; each is designed specifically to a unique set of needs, a specific program, and a distinct set of operational constraints.

Our team brings directly relevant municipal and civic expertise to the renovation of approximately 62,030 SF across two floors of the Cascade Campus. Through our partnership with the City of Aurora, we completed a comprehensive space planning and audit of the 289,000 SF Aurora Municipal Center, evaluating space utilization across multiple floors and twenty municipal departments to support growth, hybrid work, and department consolidations. We are currently collaborating with Adams County on the Western Service Center 3rd Floor Programming, focused on ADA compliance, restroom modernization, and lobby and corridor upgrades in a high-use government facility—work that closely mirrors the interior finishes, ADA improvements, and MEP upgrades contemplated in this RFQ. Combined with our work for the Town of Superior's Downtown Civic Space and our team's prior experience programming municipal facilities for projected staffing growth, we are well-equipped to translate the City's 15-year Utilities Department staffing projections into functional, adaptable space.

**Why Collab**

As a Northern Colorado–based firm, Collab Architecture has successfully completed over 30 projects in Loveland and brings a deep, working understanding of the City's codes, permitting processes, finish standards, and municipal approval procedures—familiarity that allows us to streamline coordination and proactively address challenges on a phased, occupied renovation. Our experience aligns precisely with the Cascade Campus scope: space planning and furniture layouts to City standards, ADA compliance improvements, interior finish modernization, and MEP upgrades, all delivered through the design-bid-build method this project will employ. We pair that expertise with a project team purpose-built for this work—Project Manager Kala Bailor's record of programming municipal facilities for future staffing growth, QA/QC Manager Michael Aller's four-plus decades of municipal and higher-education facility design, and trusted local engineering partners at Larsen Structural Design and Integrated MEP, whose principals have directly served the City of Loveland on Fire Stations 3 and 4 and the Larimer County Police and Courts Addition. Together, this combination of local relationships, civic renovation experience, and a collaborative, listening-first process positions Collab to deliver a thoughtful renovation that supports the Utilities Department's long-term growth while keeping existing tenants and staff safe and operational throughout construction.

---

## Project Team

### Jordan W. Lockner, AIA, NCARB
**Principal Architect & Owner**

Jordan believes that the root of all good architecture must stem from and support the community that it exists within, an ethos that directly aligns with the City of Loveland's goal of expanding the Utilities Department's Cascade Campus to serve long-term staffing growth. His ability to listen, understand, and collaborate with stakeholders while coordinating the full design process has produced successful outcomes on municipal renovations and space-planning efforts, including his role as Project Manager on the 51,500 SF Town of Windsor Public Works Campus. He is well versed in public projects throughout Northern Colorado, with deep familiarity in local codes, permitting, and the design-bid-build delivery method anticipated for this renovation. Jordan has been recognized for his leadership and mentorship with the 2022 University of Colorado ENVD Young Designer Award and a 2023 Northern Colorado 40 Under 40 honor.

*Education:* University of Colorado, B.ENVD (Architecture focus)  
*Registrations:* Licensed Architect, NCARB

**Select Experience**

- Town of Windsor, Public Works Campus - Windsor, CO (previous firm)
- City of Aurora, Municipal Center Space Planning & Remodel - Aurora, CO
- Town of Superior, Downtown Civic Space - Superior, CO
- Adams County, Western Service Center 3rd Floor Programming - Westminster, CO
- Broomfield Police Department, Evidence Storage Facility - Broomfield, CO
- Department of Public Safety, Admin & Training Facility - Windsor, CO
- Town of Estes Park, Transit Facility - Estes Park, CO
- Weld County, Grounds Building Design - Greeley, CO

---

### Kala Bailor, AIA, LEED GA
**Project Architect**

Kala has always been a person interested in both the sciences and the arts, caring deeply about the details and ensuring clients receive the best of both form and function. This attention to detail has proven successful in her role as Project Architect and Project Manager, keeping projects on schedule and within budget—skills directly relevant to coordinating a phased, 62,030 SF renovation that must keep existing City staff and tenants operating throughout construction. With successful projects ranging from the 29,000 SF Timnath Police Services Building to municipal administrative facilities and space programming for evolving operational needs, she brings demonstrated experience in programming staffing projections and stakeholder coordination. Her strong communication and coordination skills have built repeat work with clients such as the Town of Timnath, reflecting the responsiveness the City of Loveland seeks for day-to-day project management.

*Education:* University of Colorado - Denver, Master of Architecture  
*Registrations:* Licensed Architect, LEED Green Associate

**Select Experience**

- Town of Timnath, Police Services Building - Timnath, CO (previous firm)
- Town of Timnath, Town Center - Timnath, CO (previous firm)
- City of Aurora, Municipal Center Space Planning & Remodel - Aurora, CO
- Adams County, Western Service Center 3rd Floor Programming - Westminster, CO
- Town of Superior, Downtown Civic Space - Superior, CO
- Broomfield Police Department, Evidence Storage Facility - Broomfield, CO
- Eaton Public Library, Renovation & Addition - Eaton, CO

---

### Michael Aller, AIA, LEED AP
**QA/QC Manager**

With over four decades of experience in municipal and higher education facility design, Mick has successfully completed numerous projects for institutions across Colorado, bringing a depth of technical knowledge essential to a complex Utilities Department renovation involving MEP upgrades, ADA compliance, and modernization of building systems. His work has earned prestigious recognition, including the AIA Colorado Citation Award and the F.W. Dodge Silver Hard Hat Award. He is well-versed in coordinating large-scale technical projects and in ensuring documentation meets the standards of public agencies and local permitting authorities. As QA/QC Manager, Mick is dedicated to a meticulous review process that delivers high-quality, code-compliant construction documents aligned with the City of Loveland's finish and construction standards.

*Education:* University of Michigan, Master of Architecture  
*Registrations:* Licensed Architect, NCARB, LEED Accredited Professional

**Select Experience**

- City of Aurora, Municipal Center Space Planning & Remodel - Aurora, CO
- Broomfield Police Department, Evidence Storage Facility - Broomfield, CO
- Adams County, Western Service Center 3rd Floor Programming - Westminster, CO
- Town of Silverthorne, Recreation Center Expansion - Silverthorne, CO
- Town of Estes Park, Transit Facility - Estes Park, CO
- Adams County, Honnen Facility Conditions Assessment - Brighton, CO
- Department of Public Safety, Admin & Training Facility - Windsor, CO

---

## Relevant Project Experience

### CITY OF AURORA, MUNICIPAL CENTER SPACE PLANNING & REMODEL
**Aurora, CO**  |  Lead: Michael Aller, AIA, LEED AP  
*Type:* Multi-Story Interior Renovation, Space Planning & Programming  
*Services:* Architecture, Interior Design, Space Planning, Programming, Construction Administration

Collab Architecture began its partnership with the City of Aurora through a comprehensive space planning effort and audit of the 289,000-square-foot Aurora Municipal Center (AMC). This project involved evaluating space utilization across five stories and 20 municipal departments to address the facility's full capacity and prepare for future growth. By assessing operational needs, hybrid work opportunities, and potential department consolidations, our team developed a road-map to enhance efficiency, streamline workflows, and support scalability within the AMC — work directly parallel to the City of Loveland's effort to renovate approximately 62,030 SF across two floors of the Cascade Campus to accommodate the Loveland Utilities Department's 15-year staffing projections.

This project is especially relevant to the Cascade Campus renovation because it demonstrates our ability to translate programming and staffing projections into functional, phased interior renovations within a large, multi-story, actively occupied municipal building. As the AMC scope evolved, we relocated departments, established a centralized administrative hub within the building, and transformed vacated suites into purpose-built office environments featuring private offices, conference rooms, collaborative touchdown stations, and break areas — the same programmatic elements called for in the Cascade Campus scope, including office space planning to City standards, teaming spaces, and break/amenity areas.

A key outcome of this engagement was the successful consolidation and reconfiguration of multiple departments within a fully operational facility, allowing the City of Aurora to maximize the usability of existing space while planning for long-term growth — and establishing a repeat-client relationship that has produced numerous follow-on renovation projects within the AMC.

---

### ADAMS COUNTY, WESTERN SERVICE CENTER 3RD FLOOR PROGRAMMING & RENOVATION
**Westminster, CO**  |  Lead: Kala Bailor, AIA, LEED GA  
*Type:* Interior Renovation, ADA Compliance & Finish Modernization  
*Services:* Architecture, Interior Design, Programming, MEP Coordination, Construction Administration

Collab Architecture collaborated with Adams County to renovate the Western Service Center, a high-use government facility, with a focus on enhancing universal accessibility and ADA compliance. The project encompassed the transformation of restroom facilities across multiple floors into non-gender, family-friendly spaces meeting current accessibility standards for government buildings, along with the upgrade and redesign of the front lobby, elevator lobbies, and main hallways to create a more welcoming environment. This scope mirrors the Cascade Campus renovation's emphasis on ADA compliance improvements, modernization of finishes and restroom/plumbing fixtures, and the upgrade of interior public-facing spaces.

This project is directly relevant to the City of Loveland's Cascade Campus renovation because it demonstrates our team's expertise in modernizing finishes and building systems within an occupied, multi-floor municipal facility while maintaining ongoing operations — precisely the implementation strategy required at Cascade Campus, where existing tenants and select City staff will remain in the building throughout design and construction. Material selection was a critical component of the work, with ease of maintenance, long-term durability, and aesthetics balanced to suit a high-traffic public building.

The completed renovation delivered a more accessible, code-compliant, and welcoming environment for both staff and the public, and reinforced Collab's reputation for executing thoughtful, durable interior renovations for government clients — a relationship that continues today through the County's facilities team.

---

## Project Understanding & Approach

The City of Loveland's acquisition of the Cascade Campus represents a forward-thinking investment in the long-term growth of the Loveland Utilities Department, and we recognize that this renovation must thoughtfully balance immediate programmatic needs with the 15- and 30-year staffing projections that informed the City's 2023 programming efforts. Our team understands the full breadth of the scope outlined in the RFQ: the renovation of approximately 62,030 SF across the entire second floor and portions of the first floor, encompassing interior finish updates, space planning to City standards, collaborative teaming and conference spaces with audio/visual improvements, break areas and outdoor amenities, comprehensive MEP and network upgrades, ADA compliance improvements, and exterior site enhancements including parking, lighting, EV charging, and security. We are equally prepared to address the project's defining constraint—the continued occupancy of the facility by existing outside tenants and select City staff throughout both design and construction—by developing a strict, collaborative phasing and safety plan in partnership with the City's project team and selected general contractor.

Because this project will be delivered through the Design-Bid-Build method with general contractor pre-qualification, our approach emphasizes constructability, cost control, and schedule alignment at every phase, from conceptual programming through project closeout. We bring direct, relevant experience: our work on the 289,000 SF City of Aurora Municipal Center Space Planning and Remodel demonstrates our ability to assess existing multi-floor facilities, reconcile programming against staffing projections, and reconfigure office and administrative environments for evolving operational needs, while our Adams County Western Service Center renovation reflects our depth in ADA compliance and finish modernization for high-use government facilities. We will integrate independent cost estimating and value engineering at each milestone through DFH Consulting, coordinate the required one percent for the Arts public art installation with the City's Cultural Services Department, and ensure our architects and engineers—all licensed in the State of Colorado—provide the responsive communication and quality deliverables the City's evaluation criteria prioritize. With Kala Bailor serving as Project Manager and primary point of contact, supported by Jordan Lockner as Principal Architect and Michael Aller leading QA/QC, our team offers the experience, capacity, and local familiarity to deliver a renovation the Department and the City can be proud of.

**Key Challenge & Mitigation**

The single greatest challenge of this project is executing a substantial 62,030 SF renovation while existing outside tenants and select City staff remain in the occupied building throughout both design and construction. Our mitigation strategy centers on developing a detailed, phased construction sequencing plan in close collaboration with the Department, existing tenants, the City's project management staff, and the pre-qualified general contractor—establishing clear work zones, dust and noise controls, temporary access routes, and a rigorous safety plan that protects City employees, the public, and construction workers. By integrating phasing decisions early in programming and continuously coordinating through weekly OAC meetings during construction, we will maintain uninterrupted operations and services while keeping the project on schedule and within budget.

**Schedule**

Our proposed schedule aligns with the City's anticipated timeline of design commencing in Quarter 1 2026 and construction beginning in 2027, structuring a logic-driven progression from conceptual design and programming through schematic design, design development, construction documents, bidding, construction administration, and project closeout. We will refine all milestones, deliverables, and phasing checkpoints collaboratively with the City's project team to ensure an efficient and well-coordinated design process.

---

## References

**Brian Rowe, Deputy Director of Public Works, Town of Windsor**  
970.674.5400  |  browe@windsorgov.com

**Elly Watson, Business Services Manager, City of Aurora**  
303.739.7109  |  elwatson@auroragov.org

**Kyle Burg, Project Manager, Facilities & Fleet Management, Adams County**  
720.523.6062  |  KBurg@adcogov.org

---

## Fee Schedule

Our fee for the Cascade Campus Facility Renovation is structured on a unit-price, not-to-exceed (NTE) basis aligned with the City of Loveland's purchasing requirements and the Design-Bid-Build delivery method outlined in the RFQ. We establish phase-level budget caps spanning Conceptual Design and Programming, Schematic Design, Design Development, Construction Documents, Bidding and Procurement, Construction Administration, and Project Closeout, ensuring transparent cost control across the entire 62,030 SF scope. Our team provides monthly earned-value reporting that tracks completed work against each phase budget, giving Facilities and Loveland Utilities Department stakeholders clear visibility into progress, scope, and remaining contract value throughout the project.

### Standard Hourly Rates

| Role | Rate |
|------|------|
| Principal Architect / Engineer | $225 / hour |
| Project Architect / Engineer | $205 / hour |
| Project Manager / Engineer | $185 / hour |
| QA/QC Review | $185 / hour |
| CAD Technician | $115 / hour |
| Interior Designer | $95 / hour |
| Administrative | $75 / hour |

### Reimbursable Expenses

| Item | Basis |
|------|-------|
| Outside Materials / Services / Supplies | Cost + 15% |
| Mileage | $0.70 / mile |

In [17]:
# ── Open Filled IDML for Manual PDF Export ────────────────────────────────────
# Run this cell to open the IDML in InDesign, then export to PDF manually.

print("=" * 58)
print("  MANUAL EXPORT STEPS")
print("=" * 58)
print(f"\n  File: {IDML_OUTPUT}\n")
print("  1. InDesign will open with the filled template")
print("  2. File  →  Export")
print("  3. Format: Adobe PDF (Print)")
print("  4. Filename: example_rfq_to_rfp_proposal.pdf")
print("  5. Click Export → keep defaults → Export\n")

_open = subprocess.run(['open', str(IDML_OUTPUT)], capture_output=True, text=True)
if _open.returncode == 0:
    print("✓  File opened — follow the steps above in InDesign.")
else:
    print(f"Could not open automatically. Open manually:\n  {IDML_OUTPUT}")


  MANUAL EXPORT STEPS

  File: /Users/brianpak/Desktop/Desktop - Brian’s MacBook Pro/Projects/Collab Architecture/RFP_Project_2/RFP_Project_2/rfp_filled_template_v2.idml

  1. InDesign will open with the filled template
  2. File  →  Export
  3. Format: Adobe PDF (Print)
  4. Filename: example_rfq_to_rfp_proposal.pdf
  5. Click Export → keep defaults → Export

Could not open automatically. Open manually:
  /Users/brianpak/Desktop/Desktop - Brian’s MacBook Pro/Projects/Collab Architecture/RFP_Project_2/RFP_Project_2/rfp_filled_template_v2.idml


In [18]:
# ── Export PDF via Adobe InDesign ─────────────────────────────────────────────
# POSIX file must be declared OUTSIDE the tell block.
# Uses 'export format PDF type' without 'showing options' for compatibility.

_idml_str = str(IDML_OUTPUT)
_pdf_str  = str(PDF_OUTPUT)


def _try_indesign(app_name):
    script = f"""set theIDML to POSIX file \"{_idml_str}\"
tell application \"{app_name}\"
    activate
    set myDoc to open theIDML
    tell myDoc
        export format PDF type to \"{_pdf_str}\"
    end tell
    close myDoc saving no
end tell"""
    return subprocess.run(['osascript', '-e', script], capture_output=True, text=True, timeout=300)


print(f"Exporting to PDF...")
print(f"  Source IDML : {_idml_str}")
print(f"  Output PDF  : {_pdf_str}\n")

_success = False
for _app in ["Adobe InDesign 2025", "Adobe InDesign 2024", "Adobe InDesign 2023"]:
    print(f"  Trying {_app}...", end=" ", flush=True)
    _res = _try_indesign(_app)
    if _res.returncode == 0 and PDF_OUTPUT.exists():
        print(f"✓  ({PDF_OUTPUT.stat().st_size // 1024} KB)")
        subprocess.run(['open', _pdf_str])
        print(f"\n✅  PDF saved → {PDF_OUTPUT}")
        _success = True
        break
    else:
        _err = (_res.stderr or _res.stdout or "").strip()[:120]
        print(f"✗  {_err}" if _err else "✗  not found")

if not _success:
    print("\n⚠️  InDesign not found. Run the cell above to open the IDML manually.")


Exporting to PDF...
  Source IDML : /Users/brianpak/Desktop/Desktop - Brian’s MacBook Pro/Projects/Collab Architecture/RFP_Project_2/RFP_Project_2/rfp_filled_template_v2.idml
  Output PDF  : /Users/brianpak/Desktop/Desktop - Brian’s MacBook Pro/Projects/Collab Architecture/RFP_Project_2/RFP_Project_2/example_rfq_to_rfp_proposal.pdf

  Trying Adobe InDesign 2025... ✗  279:285: syntax error: Expected end of line but found identifier. (-2741)
  Trying Adobe InDesign 2024... ✗  279:285: syntax error: Expected end of line but found identifier. (-2741)
  Trying Adobe InDesign 2023... ✗  279:285: syntax error: Expected end of line but found identifier. (-2741)

⚠️  InDesign not found. Run the cell above to open the IDML manually.
